# 04 — Confounding, Selection Bias, and Bad Controls
**References:** Cinelli, Forney & Pearl (2022) "A Crash Course in Good and Bad Controls" (*Sociological Methods & Research*) · Hernán & Robins (2020) Ch. 7–9 · Elwert & Winship (2014) · Angrist & Pischke (2009) §3.2.3

## Narrative thread
```
Confounding -> Adjustment is not always good -> Colliders & selection bias -> M-bias -> Mediators & post-treatment bias -> A practical checklist
```

## "Control for everything" is wrong

Adding a covariate to a regression can **create** bias as easily as remove it. Whether a
control is good or bad is not a property of the variable — it is a property of its
**position in the causal graph**. This notebook simulates the canonical cases from
Cinelli, Forney & Pearl (2022).

| Role of $Z$ | Graph sketch | Adjust? |
|---|---|---|
| Confounder | $X \leftarrow Z \to Y$ | **Yes** — closes a backdoor |
| Mediator | $X \to Z \to Y$ | **No** — blocks part of the effect |
| Collider | $X \to Z \leftarrow Y$ | **No** — opens a spurious path |
| M-bias variable | $X \leftarrow U_1 \to Z \leftarrow U_2 \to Y$ | **No** — opens the M path |
| Outcome proxy | $Y \to Z$ | **No** — biases toward zero |
| Precision variable | $Z \to Y$ only | Yes (optional) — shrinks SEs, no bias either way |
| Instrument-like | $Z \to X$ only | Avoid — no bias removed, amplifies remaining bias |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [ ]:
# ── One simulator, seven controls: which regressions recover tau = 1? ────
n = 400_000
TAU = 1.0

def fit(df, formula):
    return smf.ols(formula, df).fit().params['x']

results = {}

# 1) Confounder: x <- z -> y
z = np.random.normal(0, 1, n)
x = z + np.random.normal(0, 1, n)
y = TAU*x + 2*z + np.random.normal(0, 1, n)
df = pd.DataFrame(dict(x=x, y=y, z=z))
results['Confounder'] = (fit(df, 'y ~ x'), fit(df, 'y ~ x + z'), 'ADJUST')

# 2) Mediator: x -> z -> y   (total effect = 1.0 = 0.4 direct + 0.6 via z)
x = np.random.normal(0, 1, n)
z = 1.0*x + np.random.normal(0, 1, n)
y = 0.4*x + 0.6*z + np.random.normal(0, 1, n)
df = pd.DataFrame(dict(x=x, y=y, z=z))
results['Mediator'] = (fit(df, 'y ~ x'), fit(df, 'y ~ x + z'), 'DO NOT')

# 3) Collider: x -> z <- y
x = np.random.normal(0, 1, n)
y = TAU*x + np.random.normal(0, 1, n)
z = x + y + np.random.normal(0, 1, n)
df = pd.DataFrame(dict(x=x, y=y, z=z))
results['Collider'] = (fit(df, 'y ~ x'), fit(df, 'y ~ x + z'), 'DO NOT')

# 4) M-bias: u1 -> x, u1 -> z, u2 -> z, u2 -> y   (z is PRE-treatment!)
u1 = np.random.normal(0, 1, n); u2 = np.random.normal(0, 1, n)
z = u1 + u2 + np.random.normal(0, 1, n)
x = u1 + np.random.normal(0, 1, n)
y = TAU*x + u2 + np.random.normal(0, 1, n)
df = pd.DataFrame(dict(x=x, y=y, z=z))
results['M-bias'] = (fit(df, 'y ~ x'), fit(df, 'y ~ x + z'), 'DO NOT')

# 5) Outcome proxy: y -> z
x = np.random.normal(0, 1, n)
y = TAU*x + np.random.normal(0, 1, n)
z = y + np.random.normal(0, 1, n)
df = pd.DataFrame(dict(x=x, y=y, z=z))
results['Proxy of Y'] = (fit(df, 'y ~ x'), fit(df, 'y ~ x + z'), 'DO NOT')

# 6) Precision variable: z -> y only
x = np.random.normal(0, 1, n)
z = np.random.normal(0, 1, n)
y = TAU*x + 2*z + np.random.normal(0, 1, n)
df = pd.DataFrame(dict(x=x, y=y, z=z))
results['Precision (z->y)'] = (fit(df, 'y ~ x'), fit(df, 'y ~ x + z'), 'optional')

print(f'True effect tau = {TAU}')
print(f'{"control type":<18} {"y ~ x":>9} {"y ~ x + z":>10}   verdict')
for k, (b0, b1, verdict) in results.items():
    print(f'{k:<18} {b0:9.3f} {b1:10.3f}   {verdict}')

Read the table column by column: for the **confounder**, only the adjusted regression is
right; for the **mediator, collider, M-structure, and outcome proxy**, adjustment *creates*
the bias. The pre-treatment timing of the M-bias variable shows that "control for anything
measured before treatment" is not a safe rule — position in the graph, not timing, decides.

## Selection bias as collider conditioning

Selecting the sample on a collider conditions on it implicitly. Classic examples:

- **Berkson's paradox** — among *hospitalized* patients, two diseases that are independent
  in the population become negatively correlated (both cause hospitalization).
- **The birthweight paradox** — among low-birthweight babies, maternal smoking appears
  *protective* for infant mortality: birthweight is a collider ($smoking \to LBW \leftarrow$
  *other, worse causes*).
- **Attractiveness vs. talent among celebrities** — fame is a collider of both.


In [ ]:
# ── Berkson's paradox: selection on a collider ───────────────────────────
n = 500_000
talent = np.random.normal(0, 1, n)
looks  = np.random.normal(0, 1, n)          # independent in the population
fame_score = talent + looks + np.random.normal(0, 0.5, n)
famous = fame_score > 2.0                    # only the "famous" enter our sample

print(f'corr(talent, looks) in the population: {np.corrcoef(talent, looks)[0,1]:+.3f}')
print(f'corr(talent, looks) among the famous:  {np.corrcoef(talent[famous], looks[famous])[0,1]:+.3f}')

fig, ax = plt.subplots(figsize=(7, 4.5))
idx = np.random.choice(n, 8000, replace=False)
ax.scatter(talent[idx], looks[idx], s=4, alpha=0.25, color='#bbbbbb', label='population')
fi = idx[famous[idx]]
ax.scatter(talent[fi], looks[fi], s=6, alpha=0.6, color='#d62728', label='selected (famous)')
ax.set_xlabel('talent'); ax.set_ylabel('looks')
ax.set_title("Berkson's paradox: independent traits become negatively\ncorrelated once you select on their common effect")
ax.legend()
plt.tight_layout(); plt.show()

## Bias amplification and the danger of near-instruments

Adjusting for a variable that strongly predicts treatment but not the outcome (a
"near-instrument") removes no confounding, but **amplifies** whatever unobserved
confounding remains (Bhattacharya & Vogt 2007; Pearl 2011). Intuition: the adjustment
soaks up the "clean" variation in $X$, so a larger *share* of what is left is confounded.


In [ ]:
# ── Bias amplification: adjusting for an instrument-like covariate ───────
n = 400_000
u = np.random.normal(0, 1, n)                # UNOBSERVED confounder
z = np.random.normal(0, 1, n)                # z -> x only (near-instrument)
x = 2.0*z + u + np.random.normal(0, 1, n)
y = 1.0*x + 2.0*u + np.random.normal(0, 1, n)
df = pd.DataFrame(dict(x=x, y=y, z=z))

b_no  = smf.ols('y ~ x', df).fit().params['x']
b_yes = smf.ols('y ~ x + z', df).fit().params['x']
print('True effect = 1.0, with unobserved confounder u present in both cases')
print(f'Without z:  bias = {b_no - 1:+.3f}')
print(f'With z:     bias = {b_yes - 1:+.3f}   <- adjusting made it WORSE')

## A practical checklist (Cinelli, Forney & Pearl 2022)

1. Draw the DAG — write down what causes treatment and what causes the outcome.
2. Adjust for variables that close backdoor paths (common causes or their proxies).
3. Never adjust for: descendants of treatment (mediators, post-treatment variables),
   colliders on the path, or proxies of the outcome.
4. Beware pre-treatment ≠ safe (M-bias) and predictive-of-treatment ≠ useful
   (bias amplification).
5. Variables that only cause $Y$ are free precision; keep them.

## Key takeaways

| Concept | Statement |
|---|---|
| Good control | Closes a backdoor path (confounder or its proxy) |
| Mediator control | Blocks the effect you are trying to measure |
| Collider control | Manufactures association out of nothing |
| Selection bias | Sampling on a collider = conditioning on it |
| M-bias | A pre-treatment variable can still be a bad control |
| Bias amplification | Near-instruments magnify unobserved confounding |
| Rule | The DAG decides — there is no model-free rule of thumb |
